In [6]:
!pip install shiny shinywidgets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.6 MB/s eta 0:00:00


In [7]:
from shiny import App, ui, reactive, render
from shinywidgets import output_widget, render_widget
import ipyleaflet
import pandas as pd

# UI
app_ui = ui.page_fluid(
    ui.panel_title("Location Viewer"),
    ui.layout_sidebar(
        ui.sidebar(
            ui.input_numeric("lat", "Latitude:", value=25.286, step=0.01),
            ui.input_numeric("lng", "Longitude:", value=-81.178, step=0.01),
            ui.input_action_button("go", "Show on Map", class_="btn-primary")
        ),
        output_widget("map", height="500px"),
        ui.hr(),
        ui.h4("Saved Locations"),
        ui.output_data_frame("table")
    )
)

# Server
def server(input, output, session):
    # Reactive value for data
    locations = reactive.Value(pd.DataFrame(columns=["ID", "Latitude", "Longitude"]))

    # Initialize Map
    m = ipyleaflet.Map(
        basemap=ipyleaflet.basemaps.Esri.WorldImagery,
        center=(25.286, -81.178),
        zoom=13
    )

    @render_widget
    def map():
        return m

    @reactive.Effect
    @reactive.event(input.go)
    def update_locations():
        # Create new row
        current_df = locations.get()
        new_id = len(current_df) + 1
        lat = input.lat()
        lng = input.lng()

        new_row = pd.DataFrame([[new_id, lat, lng]], columns=["ID", "Latitude", "Longitude"])
        updated_df = pd.concat([current_df, new_row], ignore_index=True)
        locations.set(updated_df)

        # Update Map View
        m.center = (lat, lng)
        m.zoom = 15

        # Re-draw markers
        # Filter out existing CircleMarkers to clear them (keeping other layers like tiles)
        base_layers = [l for l in m.layers if not isinstance(l, ipyleaflet.CircleMarker)]

        new_markers = []
        for idx, row in updated_df.iterrows():
            marker = ipyleaflet.CircleMarker(
                location=(row["Latitude"], row["Longitude"]),
                radius=8,
                color="#ff3333",
                fill_color="#ff3333",
                fill_opacity=0.9,
                weight=2,
                opacity=1
            )
            # Add popup
            popup_content = f"ID: {row['ID']}<br>Lat: {row['Latitude']}<br>Lng: {row['Longitude']}"
            message = ipyleaflet.HTML()
            message.value = popup_content
            marker.popup = message

            new_markers.append(marker)

        m.layers = tuple(base_layers + new_markers)

    @render.data_frame
    def table():
        return render.DataGrid(locations.get(), selection_mode="none")

app = App(app_ui, server)

In [11]:
from google.colab import output
import uvicorn

# Open the app in a new browser tab
output.serve_kernel_port_as_window(8000)

# Configure and run uvicorn directly to use the existing event loop
config = uvicorn.Config(app, port=8000, host="127.0.0.1", log_level="info")
server = uvicorn.Server(config)
await server.serve()

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

INFO:     Started server process [203]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [203]
